# IMDb dataset: data quality and exploration (PySpark)


## 1. Start Spark

`MAX_ROWS` was created for testing:

- `MAX_ROWS = 400` (test mode) reads only the first 400 rows of each IMDb file. The whole notebook runs in a few minutes, but the results are not meaningful: most joins find few or no matching rows.
- `MAX_ROWS = -1` (full run) reads every row of every file.


In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pyspark.sql import SparkSession, functions as F

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path("imdb_data")
DATA_DIR.mkdir(exist_ok=True)
SPARK_DATA_DIR = Path(os.environ.get("SPARK_DATA_DIR", "spark_data"))

MAX_ROWS = int(os.environ.get("MAX_ROWS", "400"))
RUN_NAME = "full" if MAX_ROWS < 0 else f"sample_{MAX_ROWS}"
WORK_DIR = SPARK_DATA_DIR / RUN_NAME
OUTPUT_DIR = Path("results") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPARK_DRIVER_MEMORY = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

FEATURE_MIN_RUNTIME = 70
SUCCESS_RATING = 7.5
MIN_SUCCESS_VOTES = 1000
CURRENT_YEAR = pd.Timestamp.today().year

spark = (
    SparkSession.builder
    .appName("imdb-data-quality-and-exploration")
    .master("local[*]")
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Run mode: {RUN_NAME} (MAX_ROWS = {MAX_ROWS}), results in {OUTPUT_DIR}/")


def scalar(sql):
    return spark.sql(sql).first()[0]


def export(df, filename, n=100):
    pdf = df.toPandas()
    display(pdf.head(n))
    pdf.to_csv(OUTPUT_DIR / filename, index=False)
    return pdf


def save_table(name, sql):
    path = str(WORK_DIR / "clean" / name)
    spark.sql(sql).write.mode("overwrite").parquet(path)
    spark.read.parquet(path).createOrReplaceTempView(name)


spark

## 2. Load the files into Spark

In [ ]:
TABLE_FILES = {
    "title_basics": "title.basics.tsv.gz",
    "title_ratings": "title.ratings.tsv.gz",
    "title_akas": "title.akas.tsv.gz",
    "title_crew": "title.crew.tsv.gz",
    "title_episode": "title.episode.tsv.gz",
    "title_principals": "title.principals.tsv.gz",
    "name_basics": "name.basics.tsv.gz",
}

for table, filename in TABLE_FILES.items():
    target = WORK_DIR / "raw" / table
    if not (target / "_SUCCESS").exists():
        print(f"Converting {filename} to Parquet ...")
        raw = (spark.read
            .options(sep="\t", header=True, nullValue="\\N", quote="", inferSchema=False)
            .csv(str(DATA_DIR / filename)))
        if MAX_ROWS > 0:
            raw = raw.limit(MAX_ROWS)
        raw.write.mode("overwrite").parquet(str(target))
    spark.read.parquet(str(target)).createOrReplaceTempView(table)

## 3. verify files have been loaded

In [ ]:
raw_counts = {table: spark.table(table).count() for table in TABLE_FILES}
for table, count in raw_counts.items():
    print(f"{table:18s} {count:>12,} rows")

# Data preparation

## 4. Percentage of null, missing or empty values per column

IMDb uses `\N` for missing values; the loader turns this marker into NULL.

In [ ]:
null_rows = []

for table in TABLE_FILES:
    df = spark.table(table)
    counters = [F.count(F.lit(1)).alias("__total")] + [
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
    row = df.agg(*counters).first().asDict()
    total = row["__total"]
    for c in df.columns:
        null_rows.append((table, c, total, row[c], round(100 * row[c] / total, 2) if total else None))

# let's export the percentage of null/missing/empty values for every column of every table
null_report = export(
    spark.createDataFrame(null_rows, "table string, column string, total_rows long, missing_rows long, missing_pct double")
    .orderBy(F.desc("missing_pct"), "table", "column"),
    "imdb_null_percentage_by_column.csv",
)

### Five columns with the highest percentage of null values

In [ ]:
top_5_missing = null_report.nlargest(5, "missing_pct") 
display(top_5_missing)
top_5_missing.to_csv(OUTPUT_DIR / "top_5_missing_columns.csv", index=False)

## 5. Other data-quality issues

The checks cover impossible or future dates, people credited on titles released before their birth, duplicate primary keys, out-of-range values, inconsistent series metadata, and foreign keys whose parent record is missing

In [ ]:
quality_checks = []

def add_check(category, issue, sql):
    quality_checks.append((category, issue, scalar(sql)))

add_check("Impossible dates", "People with deathYear earlier than birthYear", '''
    SELECT COUNT(*) FROM name_basics
    WHERE CAST(deathYear AS INT) < CAST(birthYear AS INT)
''')
add_check("Impossible dates", "Principals who died before they were born", '''
    SELECT COUNT(DISTINCT p.nconst) FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE CAST(n.deathYear AS INT) < CAST(n.birthYear AS INT)
''')
add_check("Impossible dates", f"People with birthYear or deathYear after {CURRENT_YEAR}", f'''
    SELECT COUNT(*) FROM name_basics
    WHERE CAST(birthYear AS INT) > {CURRENT_YEAR} OR CAST(deathYear AS INT) > {CURRENT_YEAR}
''')
add_check("Impossible dates", "Principal credits on titles released before the person's birth (archive footage excluded)", '''
    SELECT COUNT(*) FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    JOIN title_basics b ON b.tconst = p.tconst
    WHERE p.category NOT IN ('archive_footage', 'archive_sound')
      AND CAST(b.startYear AS INT) < CAST(n.birthYear AS INT)
''')
add_check("Impossible dates", "Titles with endYear earlier than startYear", '''
    SELECT COUNT(*) FROM title_basics
    WHERE CAST(endYear AS INT) < CAST(startYear AS INT)
''')
add_check("Duplicates", "Duplicate title IDs in title_basics", '''
    SELECT COUNT(*) FROM (SELECT tconst FROM title_basics GROUP BY tconst HAVING COUNT(*) > 1) d
''')
add_check("Duplicates", "Duplicate person IDs in name_basics", '''
    SELECT COUNT(*) FROM (SELECT nconst FROM name_basics GROUP BY nconst HAVING COUNT(*) > 1) d
''')
add_check("Invalid values", "Ratings outside 1-10 or not numeric", '''
    SELECT COUNT(*) FROM title_ratings
    WHERE CAST(averageRating AS DOUBLE) IS NULL OR CAST(averageRating AS DOUBLE) NOT BETWEEN 1 AND 10
''')
add_check("Invalid values", "Ratings with negative or non-numeric vote counts", '''
    SELECT COUNT(*) FROM title_ratings
    WHERE CAST(numVotes AS BIGINT) IS NULL OR CAST(numVotes AS BIGINT) < 0
''')
add_check("Invalid values", "Titles with non-positive or non-numeric runtime", '''
    SELECT COUNT(*) FROM title_basics
    WHERE runtimeMinutes IS NOT NULL AND COALESCE(CAST(runtimeMinutes AS INT), 0) <= 0
''')
add_check("Invalid values", "Titles with isAdult not in {0, 1}", '''
    SELECT COUNT(*) FROM title_basics
    WHERE isAdult IS NULL OR isAdult NOT IN ('0', '1')
''')
add_check("Invalid values", "Episodes with non-positive season/episode numbers", '''
    SELECT COUNT(*) FROM title_episode
    WHERE CAST(seasonNumber AS INT) <= 0 OR CAST(episodeNumber AS INT) <= 0
''')
add_check("Inconsistent metadata", "Episodes whose parent is not a TV series or mini-series", '''
    SELECT COUNT(*) FROM title_episode e
    JOIN title_basics b ON b.tconst = e.parentTconst
    WHERE b.titleType NOT IN ('tvSeries', 'tvMiniSeries')
''')
add_check("Missing parent record", "Ratings referencing missing titles", '''
    SELECT COUNT(*) FROM title_ratings r LEFT ANTI JOIN title_basics b ON b.tconst = r.tconst
''')
add_check("Missing parent record", "Alternative titles referencing missing titles", '''
    SELECT COUNT(*) FROM title_akas a LEFT ANTI JOIN title_basics b ON b.tconst = a.titleId
''')
add_check("Missing parent record", "Crew rows referencing missing titles", '''
    SELECT COUNT(*) FROM title_crew c LEFT ANTI JOIN title_basics b ON b.tconst = c.tconst
''')
add_check("Missing parent record", "Director credits referencing missing people", '''
    SELECT COUNT(*) FROM (
        SELECT explode(split(directors, ',')) AS nconst FROM title_crew WHERE directors IS NOT NULL
    ) d LEFT ANTI JOIN name_basics n ON n.nconst = d.nconst
''')
add_check("Missing parent record", "Writer credits referencing missing people", '''
    SELECT COUNT(*) FROM (
        SELECT explode(split(writers, ',')) AS nconst FROM title_crew WHERE writers IS NOT NULL
    ) w LEFT ANTI JOIN name_basics n ON n.nconst = w.nconst
''')
add_check("Missing parent record", "Episode rows referencing missing episode titles", '''
    SELECT COUNT(*) FROM title_episode e LEFT ANTI JOIN title_basics b ON b.tconst = e.tconst
''')
add_check("Missing parent record", "Episodes referencing missing parent series", '''
    SELECT COUNT(*) FROM title_episode e LEFT ANTI JOIN title_basics b ON b.tconst = e.parentTconst
''')
add_check("Missing parent record", "Principals referencing missing titles", '''
    SELECT COUNT(*) FROM title_principals p LEFT ANTI JOIN title_basics b ON b.tconst = p.tconst
''')
add_check("Missing parent record", "Principals referencing missing people", '''
    SELECT COUNT(*) FROM title_principals p LEFT ANTI JOIN name_basics n ON n.nconst = p.nconst
''')
add_check("Missing parent record", "knownForTitles entries referencing missing titles", '''
    SELECT COUNT(*) FROM (
        SELECT explode(split(knownForTitles, ',')) AS tconst FROM name_basics WHERE knownForTitles IS NOT NULL
    ) k LEFT ANTI JOIN title_basics b ON b.tconst = k.tconst
''')

quality_report = export(  # Data-quality issues found in the dataset and number of affected rows
    spark.createDataFrame(quality_checks, "category string, issue string, affected_rows long")
    .orderBy("category", F.desc("affected_rows")),
    "imdb_data_quality_issues.csv",
)

### Examples of the issues

In [ ]:
principals_died_before_born = spark.sql( 
    '''
    SELECT DISTINCT n.nconst, n.primaryName, n.birthYear, n.deathYear
    FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE CAST(n.deathYear AS INT) < CAST(n.birthYear AS INT)
    ORDER BY n.primaryName
''').toPandas()
display(principals_died_before_born)

display(spark.sql('''
    SELECT e.* FROM title_episode e
    LEFT ANTI JOIN title_basics b ON b.tconst = e.parentTconst
''').limit(20).toPandas())

display(spark.sql('''
    SELECT p.tconst, b.primaryTitle, b.startYear, p.nconst, n.primaryName, n.birthYear, p.category
    FROM title_principals p
    JOIN name_basics n ON n.nconst = p.nconst
    JOIN title_basics b ON b.tconst = p.tconst
    WHERE p.category NOT IN ('archive_footage', 'archive_sound')
      AND CAST(b.startYear AS INT) < CAST(n.birthYear AS INT)
''').limit(20).toPandas())

## 6. Handling the data-quality issues

The raw tables are left untouched; cleaned tables are written to Parquet with Spark and used by every analysis below.

| Issue | Handling | Reason |
|---|---|---|
| `\N` / blank strings | Stored as NULL | One consistent representation of "unknown" |
| Non-numeric numbers | `CAST` → NULL | Bad values do not stop the pipeline nor become fake zeros |
| deathYear < birthYear, future birth/death years | Both years set to NULL, `date_issue_flag = true`; the person is kept | We cannot know which year is wrong; dropping the person would orphan all their credits |
| Credits before birth | Kept, flagged with `credit_before_birth` | Often a wrong birthYear; credits are still valid |
| endYear < startYear | endYear set to NULL | startYear is the field used in analyses |
| runtime ≤ 0 | NULL | Excluded from runtime statistics |
| isAdult not 0/1 | NULL | Unknown flag rather than a guess |
| Rating outside 1–10, negative votes | Row removed from `ratings_clean` | Cannot be used in rating analyses |
| Season/episode ≤ 0 | NULL | Episode kept, season unknown |
| Duplicate primary keys | One row kept per key | Keys must be unique for joins |
| Orphan foreign keys | Rows removed from the cleaned child table (`LEFT SEMI JOIN` on the parent; counts kept in the report above) | Inner joins would drop them anyway; this makes it explicit and auditable |

Missing values are never imputed: inventing IDs, dates, ratings or runtimes would fabricate facts.

In [ ]:
DATE_ISSUE = f"COALESCE(d < b OR b > {CURRENT_YEAR} OR d > {CURRENT_YEAR}, false)"

save_table("titles_clean", '''
WITH typed AS (
    SELECT tconst, titleType, primaryTitle, originalTitle, isAdult,
           CAST(startYear AS INT) AS startYear,
           CAST(endYear AS INT) AS endYear,
           CAST(runtimeMinutes AS INT) AS runtimeMinutes,
           NULLIF(TRIM(genres), '') AS genres,
           ROW_NUMBER() OVER (PARTITION BY tconst ORDER BY primaryTitle) AS rn
    FROM title_basics
    WHERE tconst IS NOT NULL AND TRIM(tconst) <> ''
)
SELECT tconst, titleType, primaryTitle, originalTitle,
       CASE WHEN isAdult IN ('0', '1') THEN CAST(isAdult AS INT) END AS isAdult,
       startYear,
       CASE WHEN endYear >= startYear OR startYear IS NULL THEN endYear END AS endYear,
       CASE WHEN runtimeMinutes > 0 THEN runtimeMinutes END AS runtimeMinutes,
       genres
FROM typed
WHERE rn = 1
''')

save_table("people_clean", f'''
WITH typed AS (
    SELECT nconst, primaryName, primaryProfession, knownForTitles,
           CAST(birthYear AS INT) AS b, CAST(deathYear AS INT) AS d,
           ROW_NUMBER() OVER (PARTITION BY nconst ORDER BY primaryName) AS rn
    FROM name_basics
    WHERE nconst IS NOT NULL AND TRIM(nconst) <> ''
)
SELECT nconst, primaryName,
       CASE WHEN {DATE_ISSUE} THEN NULL ELSE b END AS birthYear,
       CASE WHEN {DATE_ISSUE} THEN NULL ELSE d END AS deathYear,
       {DATE_ISSUE} AS date_issue_flag,
       primaryProfession, knownForTitles
FROM typed
WHERE rn = 1
''')

save_table("ratings_clean", '''
WITH typed AS (
    SELECT r.tconst,
           CAST(r.averageRating AS DOUBLE) AS averageRating,
           CAST(r.numVotes AS BIGINT) AS numVotes,
           ROW_NUMBER() OVER (PARTITION BY r.tconst ORDER BY CAST(r.numVotes AS BIGINT) DESC) AS rn
    FROM title_ratings r
    LEFT SEMI JOIN titles_clean t ON t.tconst = r.tconst
)
SELECT tconst, averageRating, numVotes
FROM typed
WHERE rn = 1 AND averageRating BETWEEN 1 AND 10 AND numVotes >= 0
''')

save_table("principals_clean", '''
SELECT p.tconst, CAST(p.ordering AS INT) AS ordering, p.nconst, p.category, p.job, p.characters,
       COALESCE(t.startYear < n.birthYear AND p.category NOT IN ('archive_footage', 'archive_sound'), false)
           AS credit_before_birth
FROM title_principals p
JOIN titles_clean t ON t.tconst = p.tconst
JOIN people_clean n ON n.nconst = p.nconst
''')

for role, column in [("directors_clean", "directors"), ("writers_clean", "writers")]:
    save_table(role, f'''
    SELECT DISTINCT c.tconst, c.nconst
    FROM (
        SELECT tconst, explode(split({column}, ',')) AS nconst
        FROM title_crew WHERE {column} IS NOT NULL
    ) c
    LEFT SEMI JOIN titles_clean t ON t.tconst = c.tconst
    LEFT SEMI JOIN people_clean p ON p.nconst = c.nconst
    ''')

save_table("episodes_clean", '''
SELECT e.tconst, e.parentTconst,
       CASE WHEN CAST(e.seasonNumber AS INT) > 0 THEN CAST(e.seasonNumber AS INT) END AS seasonNumber,
       CASE WHEN CAST(e.episodeNumber AS INT) > 0 THEN CAST(e.episodeNumber AS INT) END AS episodeNumber
FROM title_episode e
LEFT SEMI JOIN titles_clean t ON t.tconst = e.tconst
LEFT SEMI JOIN titles_clean s ON s.tconst = e.parentTconst
''')

save_table("akas_clean", '''
SELECT a.titleId AS tconst, CAST(a.ordering AS INT) AS ordering, a.title,
       NULLIF(TRIM(a.region), '') AS region, NULLIF(LOWER(TRIM(a.language)), '') AS language,
       a.types, a.attributes, a.isOriginalTitle = '1' AS isOriginalTitle
FROM title_akas a
LEFT SEMI JOIN titles_clean t ON t.tconst = a.titleId
''')

CLEAN_TABLES = {
    "title_basics": "titles_clean", "name_basics": "people_clean", "title_ratings": "ratings_clean",
    "title_principals": "principals_clean", "title_episode": "episodes_clean", "title_akas": "akas_clean",
}
clean_counts = {clean: spark.table(clean).count() for clean in CLEAN_TABLES.values()}
cleaning_summary = pd.DataFrame([  # Row counts before and after handling the data-quality issues
    {"raw_table": raw, "clean_table": clean, "raw_rows": raw_counts[raw],
     "clean_rows": clean_counts[clean], "removed_rows": raw_counts[raw] - clean_counts[clean]}
    for raw, clean in CLEAN_TABLES.items()
])
display(cleaning_summary)
cleaning_summary.to_csv(OUTPUT_DIR / "imdb_cleaning_summary.csv", index=False)

### Verification: the issues are no longer present in the cleaned tables

In [ ]:
post_cleaning_checks = pd.DataFrame([  # Remaining number of problematic rows after cleaning (all expected to be 0)
    {"check": "deathYear < birthYear", "remaining_rows": scalar("SELECT COUNT(*) FROM people_clean WHERE deathYear < birthYear")},
    {"check": "endYear < startYear", "remaining_rows": scalar("SELECT COUNT(*) FROM titles_clean WHERE endYear < startYear")},
    {"check": "runtime <= 0", "remaining_rows": scalar("SELECT COUNT(*) FROM titles_clean WHERE runtimeMinutes <= 0")},
    {"check": "rating outside 1-10", "remaining_rows": scalar("SELECT COUNT(*) FROM ratings_clean WHERE averageRating NOT BETWEEN 1 AND 10")},
    {"check": "duplicate tconst", "remaining_rows": scalar("SELECT COUNT(*) - COUNT(DISTINCT tconst) FROM titles_clean")},
    {"check": "duplicate nconst", "remaining_rows": scalar("SELECT COUNT(*) - COUNT(DISTINCT nconst) FROM people_clean")},
    {"check": "principals without title", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM principals_clean p LEFT ANTI JOIN titles_clean t ON t.tconst = p.tconst")},
    {"check": "principals without person", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM principals_clean p LEFT ANTI JOIN people_clean n ON n.nconst = p.nconst")},
    {"check": "episodes without parent", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM episodes_clean e LEFT ANTI JOIN titles_clean t ON t.tconst = e.parentTconst")},
    {"check": "season/episode <= 0", "remaining_rows": scalar(
        "SELECT COUNT(*) FROM episodes_clean WHERE seasonNumber <= 0 OR episodeNumber <= 0")},
])
display(post_cleaning_checks)

# Data exploration

## 7. Feature-length movies with unusual runtimes by genre

A movie can belong to several genres, so it is compared with each of its genres. For every genre we compute the mean runtime and the quartiles, and flag movies outside the 1.5×IQR fences.

In [ ]:
runtime_outliers_df = spark.sql(  # Feature-length movies whose runtime is notably different from their genre average
    f'''
WITH feature_movies AS (
    SELECT tconst, primaryTitle, startYear, runtimeMinutes, explode(split(genres, ',')) AS genre
    FROM titles_clean
    WHERE titleType = 'movie' AND runtimeMinutes >= {FEATURE_MIN_RUNTIME} AND genres IS NOT NULL
),
genre_stats AS (
    SELECT genre, COUNT(*) AS genre_movie_count, AVG(runtimeMinutes) AS genre_mean,
           percentile(runtimeMinutes, 0.25) AS q1, percentile(runtimeMinutes, 0.75) AS q3
    FROM feature_movies
    GROUP BY genre
),
fenced AS (
    SELECT f.*, s.genre_mean,
           s.q1 - 1.5 * (s.q3 - s.q1) AS lower_fence,
           s.q3 + 1.5 * (s.q3 - s.q1) AS upper_fence
    FROM feature_movies f JOIN genre_stats s ON s.genre = f.genre
)
SELECT genre, tconst, primaryTitle, startYear, runtimeMinutes,
       ROUND(genre_mean, 1) AS genre_mean,
       ROUND(runtimeMinutes - genre_mean, 1) AS diff_from_mean,
       lower_fence, upper_fence,
       CASE WHEN runtimeMinutes > upper_fence THEN 'longer' ELSE 'shorter' END AS direction
FROM fenced
WHERE runtimeMinutes < lower_fence OR runtimeMinutes > upper_fence
ORDER BY genre, ABS(runtimeMinutes - genre_mean) DESC
''')
runtime_outliers_df.createOrReplaceTempView("runtime_outliers")
runtime_outliers = export(runtime_outliers_df, "feature_length_runtime_outliers_by_genre.csv")

runtime_outlier_summary = spark.sql(  # Number of runtime outliers per genre
    '''
SELECT genre, direction, COUNT(DISTINCT tconst) AS outlier_count,
       FIRST(genre_mean) AS genre_mean, FIRST(lower_fence) AS lower_fence, FIRST(upper_fence) AS upper_fence
FROM runtime_outliers
GROUP BY genre, direction
ORDER BY genre, direction
''').toPandas()
display(runtime_outlier_summary)

## 8. Top three genres per year, 1980–2025

"Top" means most movie titles released that year in the genre. Ties are kept (`RANK`), so a year may show more than three genres. Total IMDb votes and mean rating are also given.

In [ ]:
spark.sql('''
WITH movie_genres AS (
    SELECT tconst, startYear AS year, explode(split(genres, ',')) AS genre
    FROM titles_clean
    WHERE titleType = 'movie' AND startYear BETWEEN 1980 AND 2025 AND genres IS NOT NULL
),
year_genre_stats AS (
    SELECT g.year, g.genre,
           COUNT(*) AS movie_count,
           SUM(COALESCE(r.numVotes, 0)) AS total_votes,
           AVG(r.averageRating) AS mean_rating
    FROM movie_genres g
    LEFT JOIN ratings_clean r ON r.tconst = g.tconst
    GROUP BY g.year, g.genre
)
SELECT *, RANK() OVER (PARTITION BY year ORDER BY movie_count DESC) AS genre_rank
FROM year_genre_stats
''').cache().createOrReplaceTempView("year_genres")

top3_genres = export(  # Top three movie genres for each year 1980-2025
    spark.sql("SELECT * FROM year_genres WHERE genre_rank <= 3 ORDER BY year, genre_rank, genre"),
    "top_3_genres_by_year_1980_2025.csv",
)

### Top three genres compared with the other genres

For each year, the top-three group is compared with all the other genres on share of titles, titles per genre, votes per title and mean rating.

In [ ]:
spark.sql('''
WITH grouped AS (
    SELECT year, CASE WHEN genre_rank <= 3 THEN 'Top 3' ELSE 'Other genres' END AS genre_group,
           genre, movie_count, total_votes, mean_rating
    FROM year_genres
),
agg AS (
    SELECT year, genre_group,
           COUNT(DISTINCT genre) AS genre_count,
           SUM(movie_count) AS movie_count,
           SUM(total_votes) AS total_votes,
           AVG(mean_rating) AS mean_genre_rating
    FROM grouped
    GROUP BY year, genre_group
)
SELECT *,
       100 * movie_count / SUM(movie_count) OVER (PARTITION BY year) AS movie_share_pct,
       movie_count / genre_count AS movies_per_genre,
       total_votes / movie_count AS votes_per_movie
FROM agg
ORDER BY year, genre_group
''').createOrReplaceTempView("year_comparison")

year_comparison = export(  # Year-by-year comparison of the top three genres with the other genres
    spark.table("year_comparison"), "top3_vs_other_genres_by_year.csv"
)

overall_comparison = spark.sql(  # Average over 1980-2025 of the top-3 vs other genres comparison
    '''
SELECT genre_group,
       ROUND(AVG(genre_count), 2) AS genre_count,
       ROUND(AVG(movie_share_pct), 2) AS movie_share_pct,
       ROUND(AVG(movies_per_genre), 2) AS movies_per_genre,
       ROUND(AVG(votes_per_movie), 2) AS votes_per_movie,
       ROUND(AVG(mean_genre_rating), 2) AS mean_genre_rating
FROM year_comparison
GROUP BY genre_group
''').toPandas()
display(overall_comparison)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, label in zip(axes, ["movie_share_pct", "votes_per_movie", "mean_genre_rating"],
                             ["Share of genre-title records (%)", "Votes per movie", "Mean rating"]):
    sns.lineplot(data=year_comparison, x="year", y=metric, hue="genre_group", ax=ax)
    ax.set_title(label)
plt.tight_layout()
plt.show()

## 9. Directors with exactly one movie rated ≥ 8.0 and several other movies averaging ≤ 6.0

Each director of a movie gets credit for it. Among the director's rated movies, exactly one must be rated at least 8.0, and the remaining movies (at least two) must have an average rating of at most 6.0.

In [ ]:
director_results = export(  # Directors with exactly one movie rated >= 8.0 and multiple other movies averaging <= 6.0
    spark.sql('''
WITH director_movies AS (
    SELECT d.nconst, t.tconst, t.primaryTitle, t.startYear, r.averageRating, r.numVotes
    FROM directors_clean d
    JOIN titles_clean t ON t.tconst = d.tconst AND t.titleType = 'movie'
    JOIN ratings_clean r ON r.tconst = d.tconst
),
director_summary AS (
    SELECT nconst,
           SUM(CASE WHEN averageRating >= 8.0 THEN 1 ELSE 0 END) AS high_movies,
           SUM(CASE WHEN averageRating < 8.0 THEN 1 ELSE 0 END) AS other_movies,
           AVG(CASE WHEN averageRating < 8.0 THEN averageRating END) AS other_movies_avg_rating
    FROM director_movies
    GROUP BY nconst
)
SELECT s.nconst, p.primaryName,
       h.primaryTitle AS high_rated_movie, h.startYear AS high_movie_year,
       h.averageRating AS high_rating, h.numVotes AS high_movie_votes,
       s.other_movies, ROUND(s.other_movies_avg_rating, 2) AS other_movies_avg_rating
FROM director_summary s
JOIN people_clean p ON p.nconst = s.nconst
JOIN director_movies h ON h.nconst = s.nconst AND h.averageRating >= 8.0
WHERE s.high_movies = 1 AND s.other_movies >= 2 AND s.other_movies_avg_rating <= 6.0
ORDER BY h.numVotes DESC
'''),
    "directors_one_high_multiple_low.csv",
)

## 10. TV series: best episode in the first two seasons, worst episode in the last season

Only rated episodes are used. A series qualifies when at least three of its seasons have 10 or more rated episodes. The last season is the highest season number of the series.

In [ ]:
series_results = export(  # TV series with >= 3 seasons of >= 10 episodes, worst episode in last season and best in first two
    spark.sql('''
WITH eps AS (
    SELECT e.parentTconst AS series_id, s.primaryTitle AS series_title, e.seasonNumber AS season,
           e.tconst AS episode_id, ep.primaryTitle AS episode_title, r.averageRating, r.numVotes
    FROM episodes_clean e
    JOIN titles_clean s ON s.tconst = e.parentTconst AND s.titleType = 'tvSeries'
    JOIN titles_clean ep ON ep.tconst = e.tconst
    JOIN ratings_clean r ON r.tconst = e.tconst
    WHERE e.seasonNumber IS NOT NULL
),
season_sizes AS (
    SELECT series_id, season, COUNT(*) AS episode_count FROM eps GROUP BY series_id, season
),
eligible AS (
    SELECT series_id,
           SUM(CASE WHEN episode_count >= 10 THEN 1 ELSE 0 END) AS qualifying_seasons,
           MAX(season) AS last_season
    FROM season_sizes
    GROUP BY series_id
    HAVING SUM(CASE WHEN episode_count >= 10 THEN 1 ELSE 0 END) >= 3
),
ranked AS (
    SELECT eps.*, el.qualifying_seasons, el.last_season,
           ROW_NUMBER() OVER (PARTITION BY eps.series_id ORDER BY averageRating DESC, numVotes DESC) AS best_rank,
           ROW_NUMBER() OVER (PARTITION BY eps.series_id ORDER BY averageRating ASC, numVotes DESC) AS worst_rank
    FROM eps JOIN eligible el ON el.series_id = eps.series_id
)
SELECT b.series_id, b.series_title, b.qualifying_seasons, b.last_season,
       b.episode_title AS best_episode, b.season AS best_season, b.averageRating AS best_rating,
       w.episode_title AS worst_episode, w.season AS worst_season, w.averageRating AS worst_rating
FROM ranked b
JOIN ranked w ON w.series_id = b.series_id AND w.worst_rank = 1
WHERE b.best_rank = 1 AND b.season <= 2 AND w.season = w.last_season
ORDER BY b.series_title
'''),
    "series_best_early_worst_last_season.csv",
)

## 11. Actors/actresses successful in films of several languages

IMDb sets no language on `isOriginalTitle = 1` rows, so the original language of each movie is estimated from the `title.akas` records that carry the original title:

1. explicit `language` values of curated records (non-null `types`; IMDb's untyped records include machine translations), else
2. the main language of the record's country (`region`); multilingual and supranational regions are ignored.

The most frequent language wins. Movies with no evidence are left without a language.

In [ ]:
REGION_LANGUAGE = {
    "US": "en", "GB": "en", "AU": "en", "NZ": "en", "IE": "en", "CA": "en",
    "FR": "fr", "DE": "de", "AT": "de", "DDDE": "de", "XWG": "de", "IT": "it",
    "ES": "es", "MX": "es", "AR": "es", "CO": "es", "CL": "es", "PE": "es", "VE": "es", "UY": "es", "CU": "es",
    "BR": "pt", "PT": "pt", "JP": "ja", "KR": "ko", "CN": "zh", "TW": "zh", "HK": "zh",
    "RU": "ru", "SUHH": "ru", "UA": "uk", "PL": "pl", "CZ": "cs", "CSHH": "cs", "SK": "sk",
    "HU": "hu", "RO": "ro", "BG": "bg", "GR": "el", "TR": "tr", "NL": "nl", "SE": "sv",
    "NO": "no", "DK": "da", "FI": "fi", "IS": "is", "IR": "fa", "EG": "ar", "IL": "he",
    "TH": "th", "VN": "vi", "ID": "id", "HR": "hr", "RS": "sr", "SI": "sl", "EE": "et",
    "LV": "lv", "LT": "lt", "GE": "ka", "AM": "hy", "BD": "bn",
}
spark.createDataFrame(list(REGION_LANGUAGE.items()), "region string, language string") \
    .createOrReplaceTempView("region_language")

save_table("movie_language", '''
WITH evidence AS (
    SELECT a.tconst,
           CASE WHEN a.language IS NOT NULL AND a.types IS NOT NULL THEN a.language ELSE rl.language END AS language,
           CASE WHEN a.language IS NOT NULL AND a.types IS NOT NULL THEN 1 ELSE 2 END AS source_priority
    FROM akas_clean a
    JOIN titles_clean t ON t.tconst = a.tconst AND t.titleType = 'movie' AND a.title = t.originalTitle
    LEFT JOIN region_language rl ON rl.region = a.region
    WHERE NOT a.isOriginalTitle
),
votes AS (
    SELECT tconst, language, MIN(source_priority) AS source_priority, COUNT(*) AS n
    FROM evidence WHERE language IS NOT NULL
    GROUP BY tconst, language
),
ranked AS (
    SELECT tconst, language,
           ROW_NUMBER() OVER (PARTITION BY tconst ORDER BY source_priority, n DESC, language) AS rn
    FROM votes
)
SELECT tconst, language FROM ranked WHERE rn = 1
''')

language_success_df = spark.sql(  # Actors/actresses with successful films in at least two distinct languages
    f'''
WITH successful_movies AS (
    SELECT t.tconst, l.language
    FROM titles_clean t
    JOIN ratings_clean r ON r.tconst = t.tconst
    JOIN movie_language l ON l.tconst = t.tconst
    WHERE t.titleType = 'movie'
      AND r.averageRating >= {SUCCESS_RATING}
      AND r.numVotes >= {MIN_SUCCESS_VOTES}
),
actor_language AS (
    SELECT p.nconst, sm.language, COUNT(DISTINCT sm.tconst) AS movies
    FROM principals_clean p
    JOIN successful_movies sm ON sm.tconst = p.tconst
    WHERE p.category IN ('actor', 'actress')
    GROUP BY p.nconst, sm.language
)
SELECT a.nconst, n.primaryName,
       COUNT(*) AS successful_language_count,
       SUM(a.movies) AS successful_movie_count,
       concat_ws(', ', transform(
           sort_array(collect_list(named_struct('k', -a.movies, 'lang', a.language, 'm', a.movies))),
           x -> concat(x.lang, ' (', CAST(x.m AS STRING), ')'))) AS languages
FROM actor_language a
JOIN people_clean n ON n.nconst = a.nconst
GROUP BY a.nconst, n.primaryName
HAVING COUNT(*) >= 2
ORDER BY successful_language_count DESC, successful_movie_count DESC
''')
language_success = export(language_success_df, "actors_successful_in_multiple_languages.csv")

actors_3_languages = export(  # Actors/actresses with successful films in 3 or more languages
    language_success_df.where("successful_language_count >= 3"),
    "actors_successful_in_3_or_more_languages.csv",
)

## 12. Actors/actresses with 20+ year careers, 15+ movies and an early-career peak

Career span runs from the first to the last rated movie in the person's IMDb credits. The midpoint of the span splits the career into early and later halves. The person's best-rated movie must fall in the early half, and the early half's mean rating must exceed the later half's.

In [ ]:
early_peak_actors = export(  # Actors/actresses with 20+ years and 15+ movies whose best-rated movies came early in their career
    spark.sql('''
WITH actor_movies AS (
    SELECT DISTINCT p.nconst, t.tconst, t.primaryTitle, t.startYear, r.averageRating, r.numVotes
    FROM principals_clean p
    JOIN titles_clean t ON t.tconst = p.tconst AND t.titleType = 'movie' AND t.startYear IS NOT NULL
    JOIN ratings_clean r ON r.tconst = p.tconst
    WHERE p.category IN ('actor', 'actress')
),
careers AS (
    SELECT nconst, MIN(startYear) AS first_year, MAX(startYear) AS last_year, COUNT(*) AS movie_count
    FROM actor_movies
    GROUP BY nconst
    HAVING COUNT(*) >= 15 AND MAX(startYear) - MIN(startYear) >= 20
),
halves AS (
    SELECT m.*, c.first_year, c.last_year, c.movie_count,
           m.startYear <= (c.first_year + c.last_year) / 2.0 AS is_early
    FROM actor_movies m JOIN careers c ON c.nconst = m.nconst
),
best AS (
    SELECT * FROM (
        SELECT h.*, ROW_NUMBER() OVER (PARTITION BY nconst ORDER BY averageRating DESC, numVotes DESC) AS rn
        FROM halves h
    ) x WHERE rn = 1
),
means AS (
    SELECT nconst,
           AVG(CASE WHEN is_early THEN averageRating END) AS early_mean_rating,
           AVG(CASE WHEN NOT is_early THEN averageRating END) AS later_mean_rating
    FROM halves GROUP BY nconst
)
SELECT b.nconst, n.primaryName AS actor, b.first_year AS first_movie_year, b.last_year AS last_movie_year,
       b.last_year - b.first_year AS career_span_years, b.movie_count,
       b.primaryTitle AS best_movie, b.startYear AS best_movie_year, b.averageRating AS best_movie_rating,
       ROUND(m.early_mean_rating, 2) AS early_mean_rating, ROUND(m.later_mean_rating, 2) AS later_mean_rating
FROM best b
JOIN means m ON m.nconst = b.nconst
JOIN people_clean n ON n.nconst = b.nconst
WHERE b.is_early AND m.early_mean_rating > m.later_mean_rating
ORDER BY b.movie_count DESC, career_span_years DESC
'''),
    "actors_early_career_peak.csv",
)

## 13. Multi-talented film professionals and same-project performance

Director and writer credits come from `title.crew`; acting credits come from `title.principals`. A person is **multi-talented** when they have credits in at least two of the roles actor, director and writer across their movies. For these people, movies where they held several roles are compared with movies where they held only one.

In [ ]:
save_table("project_roles", '''
WITH all_roles AS (
    SELECT tconst, nconst, 'director' AS role FROM directors_clean
    UNION
    SELECT tconst, nconst, 'writer' AS role FROM writers_clean
    UNION
    SELECT tconst, nconst, 'actor' AS role FROM principals_clean WHERE category IN ('actor', 'actress')
)
SELECT a.tconst, a.nconst, t.primaryTitle, t.startYear, r.averageRating, r.numVotes,
       COUNT(*) AS role_count,
       concat_ws(' + ', array_sort(collect_list(a.role))) AS role_combination,
       MAX(CASE WHEN a.role = 'actor' THEN 1 ELSE 0 END) AS acted,
       MAX(CASE WHEN a.role = 'director' THEN 1 ELSE 0 END) AS directed,
       MAX(CASE WHEN a.role = 'writer' THEN 1 ELSE 0 END) AS wrote
FROM all_roles a
JOIN titles_clean t ON t.tconst = a.tconst AND t.titleType = 'movie'
JOIN ratings_clean r ON r.tconst = a.tconst
GROUP BY a.tconst, a.nconst, t.primaryTitle, t.startYear, r.averageRating, r.numVotes
''')

save_table("multi_talented", '''
SELECT * FROM (
    SELECT pr.nconst, p.primaryName,
           MAX(acted) + MAX(directed) + MAX(wrote) AS distinct_role_count,
           MAX(acted) = 1 AS has_acted, MAX(directed) = 1 AS has_directed, MAX(wrote) = 1 AS has_written,
           COUNT(*) AS movie_count,
           SUM(CASE WHEN role_count >= 2 THEN 1 ELSE 0 END) AS multi_role_movies,
           ROUND(AVG(CASE WHEN role_count >= 2 THEN averageRating END), 2) AS multi_role_mean_rating,
           ROUND(AVG(CASE WHEN role_count = 1 THEN averageRating END), 2) AS single_role_mean_rating
    FROM project_roles pr
    JOIN people_clean p ON p.nconst = pr.nconst
    GROUP BY pr.nconst, p.primaryName
) x
WHERE distinct_role_count >= 2
''')

multi_talented = export(  # Multi-talented film professionals (at least two of actor/director/writer)
    spark.sql("SELECT * FROM multi_talented ORDER BY distinct_role_count DESC, multi_role_movies DESC, movie_count DESC"),
    "multi_talented_film_professionals.csv",
)

performance_comparison = export(  # Ratings and votes of multi-talented people: several roles vs one role on the same movie
    spark.sql('''
SELECT CASE WHEN role_count >= 2 THEN 'Multiple roles on same movie' ELSE 'One role' END AS role_group,
       COUNT(*) AS credits, COUNT(DISTINCT tconst) AS movies,
       ROUND(AVG(averageRating), 3) AS mean_rating, percentile(averageRating, 0.5) AS median_rating,
       ROUND(AVG(numVotes), 0) AS mean_votes, percentile(numVotes, 0.5) AS median_votes
FROM project_roles pr
LEFT SEMI JOIN multi_talented m ON m.nconst = pr.nconst
GROUP BY CASE WHEN role_count >= 2 THEN 'Multiple roles on same movie' ELSE 'One role' END
'''),
    "same_project_multiple_roles_performance.csv",
)

role_combinations = export(  # Performance of multi-talented people by combination of roles held on the same movie
    spark.sql('''
SELECT role_combination, COUNT(*) AS credits,
       ROUND(AVG(averageRating), 3) AS mean_rating, percentile(averageRating, 0.5) AS median_rating,
       percentile(numVotes, 0.5) AS median_votes
FROM project_roles pr
LEFT SEMI JOIN multi_talented m ON m.nconst = pr.nconst
GROUP BY role_combination
ORDER BY credits DESC
'''),
    "role_combinations.csv",
)

paired_comparison = export(  # Within-person comparison: does the same person do better when holding several roles?
    spark.sql('''
SELECT COUNT(*) AS people_with_both_kinds_of_movies,
       ROUND(100 * AVG(CASE WHEN multi_role_mean_rating > single_role_mean_rating THEN 1 ELSE 0 END), 2)
           AS pct_better_with_multiple_roles,
       ROUND(100 * AVG(CASE WHEN multi_role_mean_rating < single_role_mean_rating THEN 1 ELSE 0 END), 2)
           AS pct_worse_with_multiple_roles,
       ROUND(AVG(multi_role_mean_rating - single_role_mean_rating), 3) AS mean_rating_difference
FROM multi_talented
WHERE multi_role_mean_rating IS NOT NULL AND single_role_mean_rating IS NOT NULL
'''),
    "multi_role_within_person_comparison.csv",
)

## 14. Summary and exported files

The result  of all the important queries are stored as .csv files :

- `imdb_null_percentage_by_column.csv`
- `top_5_missing_columns.csv`
- `imdb_data_quality_issues.csv`
- `imdb_cleaning_summary.csv`
- `feature_length_runtime_outliers_by_genre.csv`
- `top_3_genres_by_year_1980_2025.csv`
- `top3_vs_other_genres_by_year.csv`
- `directors_one_high_multiple_low.csv`
- `series_best_early_worst_last_season.csv`
- `actors_successful_in_multiple_languages.csv`
- `actors_successful_in_3_or_more_languages.csv`
- `actors_early_career_peak.csv`
- `multi_talented_film_professionals.csv`
- `same_project_multiple_roles_performance.csv`
- `role_combinations.csv`
- `multi_role_within_person_comparison.csv`
